In [0]:
# Bronze layer - load raw CSV data with audit metadata

import pyspark.sql.functions as F
from pyspark.sql.types import *

volume_path = "/Volumes/finops_catalog/focus_billing_schema/raw_ingestion_landing_volume"

print("Reading raw CSV from volume...")
raw_df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(volume_path))

initial_count = raw_df.count()
print(f"Loaded {initial_count:,} raw records")

# Add audit columns for lineage tracking
bronze_df = raw_df.withColumn(
    "bronze_ingestion_timestamp",
    F.current_timestamp()
).withColumn(
    "bronze_ingestion_date",
    F.current_date()
).withColumn(
    "source_file_path",
    F.col("_metadata.file_path")
).withColumn(
    "data_source",
    F.lit("focus_billing_csv")
).withColumn(
    "ingestion_id",
    F.lit(f"{F.current_timestamp()}")
)

print("Added audit metadata columns")

# Write to bronze table
bronze_table = "finops_catalog.focus_billing_schema.bronze_focus_billing"
print(f"\nWriting to {bronze_table}...")
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(bronze_table)
# delta.columnMapping.mode: To decouple the logical column names in the table from the physical column names stored in the actual Parquet files so that changing the name wouldnt be slow and costly!

final_count = spark.table(bronze_table).count()

print(f"\nBronze layer complete!")
print(f"  Records: {final_count:,}")
print(f"  Columns: {len(bronze_df.columns)} (added {len(bronze_df.columns) - len(raw_df.columns)} audit columns)")
print(f"  Table: {bronze_table}")
print("\nNext: Run the silver transformations for data cleansing")